# Layer deterministic checks, guidelines, and calibrated judges

This lab adapts MLflow's [custom LLM judges cookbook](https://mlflow.org/cookbook/custom-llm-judges/) without introducing direct vendor credentials or an unreviewed judge into a release gate.

Use the cheapest, most reproducible layer that can answer each question:

1. deterministic scorers for exact facts, citations, schemas, and prohibited actions;
2. separate named `Guidelines` scorers for orthogonal nuanced criteria;
3. a custom judge only when code cannot express the rubric, with human calibration and held-out validation before gating.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "examples" / "support").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open the cloned repository as your workspace.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Keep exact business rules deterministic

These rows are synthetic. The deterministic layer catches a missing source identifier and investment advice without spending judge tokens or relying on a model's interpretation.

In [ ]:
from examples.support.agent_assurance import DETERMINISTIC_CASES

assert len(DETERMINISTIC_CASES) == 3

In [ ]:
from examples.support.agent_assurance import build_deterministic_report

deterministic_report = build_deterministic_report()
print(deterministic_report.to_string(index=False))

## 2. Measure judge agreement on a held-out split

Suppose reviewers assess a nuanced rubric: whether a response explains uncertainty appropriately. Each reviewed fixture row carries the exact response text the human judged, the human verdict, and a rationale. Both judge versions are executable deterministic rules that run over those response texts — no verdict below is stored. Calibration examples may be used to revise the judge (`judge_v2` was tuned against them); validation examples must remain held out. Every human label uses the same assessment name as the judge and includes both pass and fail cases.

In [ ]:
import inspect

from examples.support.agent_assurance import judge_v1, judge_v2

print(inspect.getsource(judge_v1))
print(inspect.getsource(judge_v2))

In [ ]:
from examples.support.agent_assurance import reviewed_labels

labels = reviewed_labels()
assert labels["human_rationale"].str.len().gt(0).all()
assert set(labels.groupby("split")["human"].nunique()) == {2}
verdicts = labels[["case_id", "split", "human", "judge_v1", "judge_v2"]]
print(verdicts.to_string(index=False))

In [ ]:
from examples.support.agent_assurance import build_agreement_report

agreement_report = build_agreement_report(labels)
print("Judge-vs-human agreement, computed per split:")
print(agreement_report.to_string(index=False))

In [ ]:
from examples.support.agent_assurance import judge_disagreements

print("Where the naive keyword rule contradicts the reviewers:")
print(judge_disagreements(labels, "judge_v1").to_string(index=False))
print()
print("Where the revised rule still fails on held-out data:")
print(judge_disagreements(labels, "judge_v2").to_string(index=False))

In [ ]:
from examples.support.agent_assurance import (
    MINIMUM_TOTAL_LABELS,
    MINIMUM_VALIDATION_AGREEMENT,
    judge_authorization,
)

judge_gate = judge_authorization(labels)
validation_agreement = judge_gate["validation_agreement"]
enough_labels = len(labels) >= MINIMUM_TOTAL_LABELS
agreement_ready = validation_agreement >= MINIMUM_VALIDATION_AGREEMENT
judge_gate_authorized = enough_labels and agreement_ready
print(
    f"labels: {len(labels)} of {MINIMUM_TOTAL_LABELS} required "
    f"-> enough_labels={enough_labels}"
)
print(
    f"validation agreement: {validation_agreement:.2f} "
    f"(minimum {MINIMUM_VALIDATION_AGREEMENT}) -> agreement_ready={agreement_ready}"
)
print(f"authorized: {judge_gate_authorized} -> status {judge_gate['judge_status']!r}")

## 3. Optional connected custom judge

The connected path resolves the approved logical `judge-model` to a keyless `endpoints:/...` URI. `make_judge` instructions may use only reserved variables such as `{{ inputs }}`, `{{ outputs }}`, `{{ expectations }}`, `{{ conversation }}`, and `{{ trace }}`.

Register and version the judge, attach human feedback with source `group:domain-reviewers`, inspect every rationale and scorer error, and repeat the same calibration/validation measurement. Never place an individual reviewer email in tags or assessment provenance.

In [ ]:
from examples.support.agent_assurance import run_connected_custom_judge

RUN_CONNECTED_CUSTOM_JUDGE = False
JUDGE_MODEL_URI = None
if RUN_CONNECTED_CUSTOM_JUDGE:
    if not JUDGE_MODEL_URI:
        raise ValueError("Resolve the governed judge model first")
    print(
        run_connected_custom_judge(
            labels, deterministic_report, judge_model_uri=JUDGE_MODEL_URI
        )
    )
else:
    print("CONNECTED CUSTOM JUDGE SKIPPED")

## Result

Computed on the held-out validation split, the revised rule agrees with reviewers 0.75 of the time versus 0.50 for the naive keyword rule, and the disagreement rows show exactly where hedge-word counting fails. `judge_v2` reaches the illustrative agreement threshold, but twelve labels are far below the required fifty, so authorization stays report-only. The deterministic fact, citation, and policy checks continue to gate every critical row.